# Demo v0.1 — обратный расчёт ТЭП

Три сценария:
1. Малый квартал — ограничивающий фактор виден сразу.
2. Средний квартал с ППТ.
3. Сравнение с/без ППТ на крупном квартале.

In [ ]:
from urban_model.normatives import load_normatives
from urban_model.models import Site, CalculationOptions
from urban_model.modes.inverse import solve_max_kit
import pandas as pd

norms = load_normatives('spb')
norms.profile

## Сценарий 1. Малый квартал 1 га

In [ ]:
res1 = solve_max_kit(Site(area_m2=10_000, name='Малый'), CalculationOptions(floors=10), norms)
print(res1.summary())

## Сценарий 2. Средний квартал 3 га, с ППТ

In [ ]:
res2 = solve_max_kit(Site(area_m2=30_000, name='Средний'), CalculationOptions(floors=12, planning_doc=True), norms)
print(res2.summary())

## Аудит-трейл: формулы, источники, статусы

In [ ]:
rows = []
for name, fld in res2.model_dump().items():
    if isinstance(fld, dict) and 'value' in fld:
        rows.append({
            'поле': name,
            'значение': fld['value'],
            'ед.': fld.get('unit'),
            'статус': fld.get('status'),
            'источник': fld.get('source'),
            'формула': fld.get('formula'),
        })
df = pd.DataFrame(rows)
df

## Сценарий 3. Крупный квартал 10 га — сравнение с/без ППТ

In [ ]:
site = Site(area_m2=100_000, name='Крупный')
r_yes = solve_max_kit(site, CalculationOptions(floors=15, planning_doc=True), norms)
r_no  = solve_max_kit(site, CalculationOptions(floors=15, planning_doc=False), norms)

comp = pd.DataFrame({
    'с ППТ':    [r_yes.kit.value, r_yes.apartments_area.value, r_yes.population.value, r_yes.balance.surplus, r_yes.limiting_factor],
    'без ППТ':  [r_no.kit.value, r_no.apartments_area.value, r_no.population.value, r_no.balance.surplus, r_no.limiting_factor],
}, index=['КИТ', 'площадь квартир, м²', 'население, чел', 'резерв, м²', 'ограничивающий фактор'])
comp